# QueryOrchestratorAgent Testing Notebook

This notebook provides comprehensive testing of the QueryOrchestratorAgent using real OpenAI API calls.
**No mocking** - this tests the actual agent behavior with live API responses.

## Prerequisites
- Set `OPENAI_API_KEY` environment variable
- Ensure all dependencies are installed

## Test Categories
1. **Basic Functionality** - Core agent operations
2. **Golden Query Dataset** - Predefined test cases
3. **Edge Cases** - Error handling and boundary conditions
4. **Performance Testing** - Timing and concurrent execution
5. **Interactive Testing** - Manual query testing

In [ ]:
# Setup and imports
import os
import sys
import json
import asyncio
import time
from datetime import datetime
from typing import List, Dict, Any

# Add the backend directory to Python path
import os
backend_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(backend_path)
print(f"Added to Python path: {backend_path}")

# Import agent and state management
from app.agents.query_orchestrator_agent import QueryOrchestratorAgent, create_query_orchestrator_agent
from app.agents.state import (
    SmartShopperState, 
    SearchQuery, 
    create_initial_state,
    get_state_summary,
    add_agent_step
)

# Check API key
if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set! Please set it before running tests.")
    print("Example: export OPENAI_API_KEY='your-api-key-here'")
else:
    print("OPENAI_API_KEY found")

print("Imports successful")

## 1. Basic Functionality Tests

In [ ]:
# Test 1: Agent Initialization
print("Test 1: Agent Initialization")
print("=" * 50)

try:
    agent = QueryOrchestratorAgent()
    print(f"Agent created successfully")
    print(f"   Name: {agent.name}")
    print(f"   Color: {agent.color}")
    print(f"   LLM: {type(agent.llm).__name__}")
    print(f"   Prompt: {type(agent.prompt).__name__}")
except Exception as e:
    print(f"Agent initialization failed: {e}")

print("\n")

In [ ]:
# Test 2: Simple Query Processing
print("Test 2: Simple Query Processing")
print("=" * 50)

async def test_simple_query():
    test_query = "gaming laptop under $2000"
    print(f"Query: '{test_query}'")
    
    # Create initial state
    state = create_initial_state(
        raw_query=test_query,
        user_id="test_user",
        run_id="simple_test_001"
    )
    
    print(f"Initial state created: {state['run_id']}")
    
    # Process query
    start_time = time.time()
    result_state = await agent.process(state)
    end_time = time.time()
    
    processing_time = (end_time - start_time) * 1000
    print(f"Processing time: {processing_time:.2f}ms")
    
    # Check results
    if result_state["search_query"]:
        search_query = result_state["search_query"]
        print("Search query created successfully")
        print(f"   Raw query: {search_query.raw_query}")
        print(f"   Normalized: {search_query.normalized_query}")
        print(f"   Intent: {search_query.intent}")
        print(f"   Category: {search_query.category}")
        print(f"   Budget max: {search_query.budget_max}")
        print(f"   Constraints: {search_query.constraints}")
        print(f"   Priorities: {search_query.priorities}")
        
        # Check Tavily parameters
        if "tavily_search_params" in result_state:
            tavily_params = result_state["tavily_search_params"]
            print("Tavily parameters generated")
            print(f"   Params: {json.dumps(tavily_params, indent=2)}")
        
        # Check agent steps
        if result_state["agent_steps"]:
            step = result_state["agent_steps"][0]
            print("Agent execution tracked")
            print(f"   Status: {step.status}")
            print(f"   Execution time: {step.execution_time_ms}ms")
            print(f"   Items processed: {step.items_processed}")
    else:
        print("No search query created")
        if result_state["errors"]:
            print(f"   Errors: {result_state['errors']}")
    
    return result_state

# Run the test
simple_result = await test_simple_query()
print("\n")

## 2. Golden Query Dataset Testing

Test with predefined queries that cover different intents and categories.

In [ ]:
# Test 3: Golden Query Dataset
print("Test 3: Golden Query Dataset")
print("=" * 50)

golden_queries = [
    {
        "query": "iPhone 15 Pro Max 256GB",
        "expected": {
            "intent": "product_search",
            "category": "smartphone",
            "has_brand": True
        }
    },
    {
        "query": "best wireless headphones under $200 2024",
        "expected": {
            "intent": "product_search", 
            "category": "headphones",
            "has_budget": True
        }
    },
    {
        "query": "MacBook Pro vs Dell XPS 15 comparison",
        "expected": {
            "intent": "comparison",
            "category": "laptop"
        }
    },
    {
        "query": "Sony WH-1000XM5 review",
        "expected": {
            "intent": "review_search",
            "category": "headphones",
            "has_brand": True
        }
    },
    {
        "query": "4K gaming monitor 27 inch under 500 euros",
        "expected": {
            "intent": "product_search",
            "category": "monitor",
            "has_budget": True
        }
    },
    {
        "query": "budget Android phone for students",
        "expected": {
            "intent": "product_search",
            "category": "smartphone"
        }
    }
]

async def test_golden_queries():
    results = []
    
    for i, test_case in enumerate(golden_queries):
        print(f"\nTest {i+1}: '{test_case['query']}'")
        print("-" * 60)
        
        # Create state and process
        state = create_initial_state(
            raw_query=test_case["query"],
            run_id=f"golden_test_{i+1:03d}"
        )
        
        start_time = time.time()
        try:
            result_state = await agent.process(state)
            processing_time = (time.time() - start_time) * 1000
            
            if result_state["search_query"]:
                search_query = result_state["search_query"]
                
                # Check expectations
                expected = test_case["expected"]
                
                # Intent check
                intent_match = search_query.intent == expected["intent"]
                print(f"   Intent: {search_query.intent} {'' if intent_match else ''} (expected: {expected['intent']})")
                
                # Category check
                if "category" in expected:
                    category_match = search_query.category and expected["category"].lower() in search_query.category.lower()
                    print(f"   Category: {search_query.category} {'' if category_match else ''} (expected: {expected['category']})")
                
                # Brand check
                if expected.get("has_brand"):
                    has_brand = search_query.brand is not None
                    print(f"   Brand: {search_query.brand} {'' if has_brand else ''} (expected: present)")
                
                # Budget check
                if expected.get("has_budget"):
                    has_budget = search_query.budget_max is not None or search_query.budget_min is not None
                    budget_info = f"min:{search_query.budget_min}, max:{search_query.budget_max}"
                    print(f"   Budget: {budget_info} {'' if has_budget else ''} (expected: present)")
                
                print(f"   Normalized: {search_query.normalized_query}")
                print(f"   Constraints: {search_query.constraints}")
                print(f"   Processing: {processing_time:.2f}ms")
                
                results.append({
                    "query": test_case["query"],
                    "intent": search_query.intent,
                    "category": search_query.category,
                    "brand": search_query.brand,
                    "budget_max": search_query.budget_max,
                    "processing_time_ms": processing_time,
                    "success": True
                })
            else:
                print(f"   Failed to parse query")
                if result_state["errors"]:
                    print(f"   Errors: {result_state['errors']}")
                results.append({
                    "query": test_case["query"],
                    "success": False,
                    "errors": result_state.get("errors", [])
                })
                
        except Exception as e:
            print(f"   Exception: {e}")
            results.append({
                "query": test_case["query"],
                "success": False,
                "exception": str(e)
            })
    
    return results

# Run golden query tests
golden_results = await test_golden_queries()

# Summary
print("\n" + "=" * 60)
print("GOLDEN QUERIES SUMMARY")
print("=" * 60)
successful = sum(1 for r in golden_results if r.get("success", False))
total = len(golden_results)
print(f"Success rate: {successful}/{total} ({successful/total*100:.1f}%)")

if successful > 0:
    avg_time = sum(r.get("processing_time_ms", 0) for r in golden_results if r.get("success")) / successful
    print(f"Average processing time: {avg_time:.2f}ms")

print("\n")

## 3. Edge Cases and Error Handling

In [ ]:
# Test 4: Edge Cases
print("Test 4: Edge Cases and Error Handling")
print("=" * 50)

edge_cases = [
    "",  # Empty query
    "   ",  # Whitespace only
    "a",  # Single character
    "buy stuff",  # Very vague
    "gaming laptop ",  # With emojis
    "Ich möchte einen Gaming-Laptop unter 2000€",  # German
    "x" * 500,  # Very long query
    "123 456 789",  # Numbers only
    "!@#$%^&*()",  # Special characters only
]

async def test_edge_cases():
    edge_results = []
    
    for i, query in enumerate(edge_cases):
        print(f"\nEdge Case {i+1}: '{query[:50]}{'...' if len(query) > 50 else ''}'")
        print("-" * 60)
        
        state = create_initial_state(
            raw_query=query,
            run_id=f"edge_test_{i+1:03d}"
        )
        
        try:
            start_time = time.time()
            result_state = await agent.process(state)
            processing_time = (time.time() - start_time) * 1000
            
            if result_state["search_query"]:
                search_query = result_state["search_query"]
                print(f"   Processed successfully")
                print(f"   Intent: {search_query.intent}")
                print(f"   Normalized: '{search_query.normalized_query}'")
                print(f"   Category: {search_query.category}")
                print(f"   Processing: {processing_time:.2f}ms")
                
                edge_results.append({
                    "query": query,
                    "intent": search_query.intent,
                    "normalized": search_query.normalized_query,
                    "processing_time_ms": processing_time,
                    "success": True
                })
            else:
                print(f"   Failed to create search query")
                if result_state["errors"]:
                    print(f"   Errors: {result_state['errors']}")
                edge_results.append({
                    "query": query,
                    "success": False,
                    "errors": result_state.get("errors", [])
                })
                
        except Exception as e:
            print(f"   Exception: {e}")
            edge_results.append({
                "query": query,
                "success": False,
                "exception": str(e)
            })
    
    return edge_results

# Run edge case tests
edge_results = await test_edge_cases()

# Summary
print("\n" + "=" * 60)
print("EDGE CASES SUMMARY")
print("=" * 60)
handled = sum(1 for r in edge_results if r.get("success", False))
total = len(edge_results)
print(f"Successfully handled: {handled}/{total} ({handled/total*100:.1f}%)")
print("Note: It's normal for some edge cases to fail gracefully")

print("\n")

## 4. Performance Testing

In [ ]:
# Test 5: Performance and Concurrent Execution
print("Test 5: Performance Testing")
print("=" * 50)

async def test_performance():
    print("\nConcurrent Execution Test")
    print("-" * 40)
    
    # Test queries for concurrent execution
    concurrent_queries = [
        "laptop for programming",
        "wireless mouse under $50",
        "4K monitor 32 inch",
        "gaming keyboard mechanical",
        "smartphone with good camera"
    ]
    
    async def process_single_query(query: str, query_id: int):
        state = create_initial_state(
            raw_query=query,
            run_id=f"perf_test_{query_id:03d}"
        )
        
        start_time = time.time()
        result_state = await agent.process(state)
        end_time = time.time()
        
        return {
            "query": query,
            "query_id": query_id,
            "processing_time_ms": (end_time - start_time) * 1000,
            "success": result_state["search_query"] is not None,
            "intent": result_state["search_query"].intent if result_state["search_query"] else None
        }
    
    # Run concurrent tasks
    start_time = time.time()
    tasks = [
        process_single_query(query, i) 
        for i, query in enumerate(concurrent_queries)
    ]
    
    results = await asyncio.gather(*tasks)
    total_time = (time.time() - start_time) * 1000
    
    print(f"Total concurrent execution time: {total_time:.2f}ms")
    print(f"Number of concurrent queries: {len(concurrent_queries)}")
    
    # Individual results
    for result in results:
        status = "" if result["success"] else ""
        print(f"   {status} '{result['query']}' - {result['processing_time_ms']:.2f}ms - {result['intent']}")
    
    # Performance metrics
    successful_results = [r for r in results if r["success"]]
    if successful_results:
        avg_time = sum(r["processing_time_ms"] for r in successful_results) / len(successful_results)
        min_time = min(r["processing_time_ms"] for r in successful_results)
        max_time = max(r["processing_time_ms"] for r in successful_results)
        
        print(f"\nPerformance Metrics:")
        print(f"   Average processing time: {avg_time:.2f}ms")
        print(f"   Fastest query: {min_time:.2f}ms")
        print(f"   Slowest query: {max_time:.2f}ms")
        print(f"   Success rate: {len(successful_results)}/{len(results)} ({len(successful_results)/len(results)*100:.1f}%)")
    
    return results

# Run performance tests
perf_results = await test_performance()
print("\n")

## 5. Interactive Testing

Test custom queries interactively.

In [ ]:
# Test 6: Interactive Query Testing
print("Test 6: Interactive Query Testing")
print("=" * 50)

async def test_interactive_query(query: str):
    """Test a single interactive query with detailed output"""
    print(f"\nTesting Query: '{query}'")
    print("-" * 60)
    
    # Create state
    state = create_initial_state(
        raw_query=query,
        user_id="interactive_user",
        run_id=f"interactive_{int(time.time())}"
    )
    
    # Process with timing
    start_time = time.time()
    result_state = await agent.process(state)
    processing_time = (time.time() - start_time) * 1000
    
    # Display results
    if result_state["search_query"]:
        search_query = result_state["search_query"]
        
        print("Parsing successful!")
        print(f"\nParsed Results:")
        print(f"   Raw Query: {search_query.raw_query}")
        print(f"   Normalized: {search_query.normalized_query}")
        print(f"   Intent: {search_query.intent}")
        print(f"   Category: {search_query.category or 'Not specified'}")
        print(f"   Brand: {search_query.brand or 'Not specified'}")
        print(f"   Budget Range: ${search_query.budget_min or '?'} - ${search_query.budget_max or '?'}")
        print(f"   Region: {search_query.region}")
        
        if search_query.constraints:
            print(f"   Constraints: {', '.join(search_query.constraints)}")
        else:
            print(f"   Constraints: None")
            
        if search_query.priorities:
            print(f"   Priorities: {', '.join(search_query.priorities)}")
        else:
            print(f"   Priorities: None")
        
        # Show Tavily parameters
        if "tavily_search_params" in result_state and result_state["tavily_search_params"]:
            print(f"\nTavily Search Parameters:")
            for key, value in result_state["tavily_search_params"].items():
                print(f"   {key}: {value}")
        
        # Show execution info
        if result_state["agent_steps"]:
            step = result_state["agent_steps"][0]
            print(f"\n⏱Execution Info:")
            print(f"   Status: {step.status}")
            print(f"   Processing Time: {step.execution_time_ms}ms")
            print(f"   Items Processed: {step.items_processed}")
            print(f"   Cost: ${step.cost_usd:.4f}")
        
        # Show state summary
        summary = get_state_summary(result_state)
        print(f"\nState Summary:")
        print(f"   Run ID: {summary['run_id']}")
        print(f"   Agents Completed: {summary['progress']['agents_completed']}")
        print(f"   Total Cost: ${summary['progress']['total_cost_usd']:.4f}")
        print(f"   Errors: {summary['progress']['errors']}")
        
    else:
        print("Parsing failed!")
        if result_state["errors"]:
            print(f"\nErrors:")
            for error in result_state["errors"]:
                print(f"   - {error}")
    
    print(f"\n⏱Total Processing Time: {processing_time:.2f}ms")
    return result_state

# Pre-defined interactive test queries
interactive_test_queries = [
    "best laptop for machine learning under 3000 dollars",
    "iPhone 15 vs Samsung Galaxy S24 Ultra camera comparison",
    "Sony WH-1000XM5 headphones review and alternatives",
    "budget 4K TV 55 inch for gaming",
    "premium mechanical keyboard for programming"
]

print("\nRunning pre-defined interactive tests...")
print("=" * 60)

interactive_results = []
for query in interactive_test_queries:
    result = await test_interactive_query(query)
    interactive_results.append(result)
    print("\n" + "=" * 60)

print("\nAll interactive tests completed!")

## 6. Manual Testing Cell

Use this cell to test your own custom queries.

In [ ]:
# Manual Testing - Edit the query below and run this cell
print("Manual Query Testing")
print("=" * 50)

# Test query
manual_query = "gaming laptop with RTX 4070 under 2500 euros for university"

print(f"Testing your query: '{manual_query}'")
print("(You can edit the manual_query variable above to test different queries)")

# Test the manual query
manual_result = await test_interactive_query(manual_query)

print("\nTo test another query:")
print("1. Edit the 'manual_query' variable above")
print("2. Run this cell again")

## 7. Overall Test Summary

In [ ]:
# Overall Summary
print("OVERALL TEST SUMMARY")
print("=" * 60)

# Collect all test results
all_results = []
if 'golden_results' in locals():
    all_results.extend(golden_results)
if 'edge_results' in locals():
    all_results.extend(edge_results)
if 'perf_results' in locals():
    all_results.extend(perf_results)

# Calculate overall metrics
total_tests = len(all_results)
successful_tests = sum(1 for r in all_results if r.get("success", False))
failed_tests = total_tests - successful_tests

if total_tests > 0:
    success_rate = (successful_tests / total_tests) * 100
    
    print(f"Total Tests Run: {total_tests}")
    print(f"Successful: {successful_tests} ({success_rate:.1f}%)")
    print(f"Failed: {failed_tests} ({100-success_rate:.1f}%)")
    
    # Performance metrics from successful tests
    successful_with_time = [r for r in all_results if r.get("success") and "processing_time_ms" in r]
    if successful_with_time:
        avg_time = sum(r["processing_time_ms"] for r in successful_with_time) / len(successful_with_time)
        min_time = min(r["processing_time_ms"] for r in successful_with_time)
        max_time = max(r["processing_time_ms"] for r in successful_with_time)
        
        print(f"\n⏱Performance Metrics:")
        print(f"   Average Response Time: {avg_time:.2f}ms")
        print(f"   Fastest Response: {min_time:.2f}ms")
        print(f"   Slowest Response: {max_time:.2f}ms")
    
    # Intent distribution
    intents = {}
    for r in all_results:
        if r.get("success") and "intent" in r:
            intent = r["intent"]
            intents[intent] = intents.get(intent, 0) + 1
    
    if intents:
        print(f"\nIntent Distribution:")
        for intent, count in sorted(intents.items()):
            print(f"   {intent}: {count} queries")
    
    # Category distribution
    categories = {}
    for r in all_results:
        if r.get("success") and "category" in r and r["category"]:
            category = r["category"]
            categories[category] = categories.get(category, 0) + 1
    
    if categories:
        print(f"\nCategory Distribution:")
        for category, count in sorted(categories.items()):
            print(f"   {category}: {count} queries")
    
    # Test quality assessment
    print(f"\nTest Quality Assessment:")
    if success_rate >= 90:
        print("   EXCELLENT - Agent performs very well across all test cases")
    elif success_rate >= 80:
        print("   GOOD - Agent handles most queries correctly")
    elif success_rate >= 70:
        print("   ACCEPTABLE - Agent works for common cases but needs improvement")
    else:
        print("   NEEDS WORK - Agent requires significant improvements")
    
    if avg_time < 2000:
        print("   FAST - Response times are excellent")
    elif avg_time < 5000:
        print("   MODERATE - Response times are acceptable")
    else:
        print("   SLOW - Response times may need optimization")

else:
    print("No test results available")

print("\n" + "=" * 60)
print("QueryOrchestratorAgent testing completed!")
print("Review the results above to assess agent performance.")
print("=" * 60)

## Notes and Recommendations

### What This Notebook Tests:
1. **Agent Initialization** - Verifies the agent can be created properly
2. **Query Parsing** - Tests LLM-based query understanding and structuring
3. **Intent Recognition** - Validates product_search, review_search, and comparison intents
4. **Category Detection** - Tests category identification (laptop, smartphone, etc.)
5. **Budget Extraction** - Verifies price range parsing
6. **Brand Recognition** - Tests brand name identification
7. **Constraint Analysis** - Validates constraint extraction (gaming, wireless, etc.)
8. **Tavily Parameter Generation** - Tests search parameter creation
9. **Error Handling** - Tests graceful handling of edge cases
10. **Performance** - Measures response times and concurrent execution

### Expected Performance Targets:
- **Success Rate**: >85% for normal queries
- **Response Time**: <3 seconds per query
- **Intent Accuracy**: >90% for clear queries
- **Category Accuracy**: >80% for product queries

### Next Steps:
1. If tests fail, check your OPENAI_API_KEY setup
2. Review failed cases to improve prompt engineering
3. Consider adding more golden test cases for edge cases
4. Integrate with TavilyRetrieverAgent for end-to-end testing
5. Add cost tracking for OpenAI API usage